# The Alice and Bob Age Problem

In this notebook we are going to solve the following text problem using the constraint solver `Z3`.
<ul>
<li>    Alice is twice as old as Bob was when Alice was as old as Bob is now.</li>
<li>    When Bob is as old as Alice is now, the sum of their ages will be 90.</li>
    
    How old are Alice and Bob now?

</ul>

---
### Solution

In order to solve this problem, we have to formalise using first order logic.  Our first task is to choose 
appropriate variables:
 * `A` is the age of Alice,
 * `B` is the age of Bob,
 * `D` is the age difference between Alice and Bob:
   $$ \texttt{D} = \texttt{A} - \texttt{B} $$

   Obviously, `D` is redundant since it can be computed from `A` and `B`.  Nevertheless, using `D` 
   simplifies the formalisation of the problem.
 * `Y` is the year Alice was born.
 
   Again, `Y` is redundant and the exercise does not contain enough information to determine `Y`.
   Still, the use of `Y` has helped me to get a grasp of the problem.
   
Use these variables and try to put numbers on the different events:

"your comments here"


First, we initialize the `z3-solver` library and create the asynchronous context.

In [1]:
import { init } from 'z3-solver';
const { Context } = await init();
const Z3 = Context("main");

We declare our constraint variables for Alice's and Bob's current ages. Since ages in these types of riddles are typically whole numbers, we use `Z3.Int.const`.

In [2]:
const A = Z3.Int.const('Alice');
const B = Z3.Int.const('Bob');

Next, we create the *solver* object and define the time difference between their ages. We will assume Alice is older based on the phrasing.

In [3]:
const S = new Z3.Solver();

In [4]:
S.add(A.mul(2).eq(B.sub(A.sub(B))));   
S.add(A.add(A.add(A.sub(B))).eq(90)); 

We run the solver to check for satisfiability and extract the model.

In [5]:
await S.check();
const model = S.model();
model.toString();

(define-fun Alice () Int
  60)
(define-fun Bob () Int
  90)


Finally, we evaluate the variables, convert them to standard JavaScript numbers, and output the solution.

In [6]:
const ageAlice = parseInt(model.eval(A).toString());
const ageBob   = parseInt(model.eval(B).toString());

In [7]:
console.log(`Alice is ${ageAlice} years old.`);
console.log(`Bob is ${ageBob} years old.`);
console.log(`\nVerification:`);
console.log(`The age difference is ${ageAlice - ageBob} years.`);
console.log(`When Alice was ${ageBob} (${ageAlice - ageBob} years ago), Bob was ${ageBob - (ageAlice - ageBob)}.`);
console.log(`Is Alice's age (${ageAlice}) twice Bob's past age (${ageBob - (ageAlice - ageBob)})? Yes.`);
console.log(`When Bob is ${ageAlice} (in ${ageAlice - ageBob} years), Alice will be ${ageAlice + (ageAlice - ageBob)}.`);
console.log(`Will the sum be 90? ${ageAlice} + ${ageAlice + (ageAlice - ageBob)} = ${ageAlice * 2 + (ageAlice - ageBob)} = 90. Yes.`);

Alice is 60 years old.
Bob is 90 years old.

Verification:
The age difference is -30 years.
When Alice was 90 (-30 years ago), Bob was 120.
Is Alice's age (60) twice Bob's past age (120)? Yes.
When Bob is 60 (in -30 years), Alice will be 30.
Will the sum be 90? 60 + 30 = 90 = 90. Yes.
